# DP-PEFT Results Analysis

This notebook generates all paper-ready figures for the differential privacy placement study.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.5)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'

## Load Results

In [ ]:
results_dir = Path('../results')

def load_all_results(model='bert', dataset='agnews'):
    results = []
    for file in results_dir.glob(f'{model}_{dataset}_*.json'):
        with open(file, 'r') as f:
            data = json.load(f)
            data['filename'] = file.stem
            results.append(data)
    return pd.DataFrame(results)

df = load_all_results('bert', 'agnews')
df.head()

## Figure 1: Privacy-Utility Curves

In [ ]:
placements = ['no_dp', 'full_dp', 'last_layer', 'adapter_only', 'head_adapter', 'partial_backbone']
placement_labels = {
    'no_dp': 'No DP (Baseline)',
    'full_dp': 'Full-Model DP',
    'last_layer': 'Last-Layer DP',
    'adapter_only': 'Adapter-Only DP',
    'head_adapter': 'Head+Adapter DP',
    'partial_backbone': 'Partial Backbone DP'
}

fig, ax = plt.subplots(figsize=(12, 7))

for placement in placements:
    placement_data = df[df['filename'].str.contains(placement)].sort_values('final_epsilon')
    
    epsilons = placement_data['final_epsilon'].values
    accuracies = placement_data['final_accuracy'].values
    
    ax.plot(epsilons, accuracies, marker='o', linewidth=2, markersize=8, 
            label=placement_labels[placement])

ax.set_xlabel('Privacy Budget (ε)', fontsize=14)
ax.set_ylabel('Test Accuracy', fontsize=14)
ax.set_title('Privacy-Utility Tradeoff Across DP Placements', fontsize=16, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xscale('log')

plt.savefig('../results/figures/privacy_utility_curve.pdf')
plt.savefig('../results/figures/privacy_utility_curve.png')
plt.show()

## Figure 2: Convergence Curves

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

epsilon = 1.0
for placement in placements:
    placement_data = df[(df['filename'].str.contains(placement)) & 
                        (df['final_epsilon'].between(epsilon-0.1, epsilon+0.1))]
    
    if len(placement_data) > 0:
        convergence = placement_data.iloc[0]['convergence_curve']
        ax.plot(convergence, linewidth=2, label=placement_labels[placement])

ax.set_xlabel('Epoch', fontsize=14)
ax.set_ylabel('Loss', fontsize=14)
ax.set_title(f'Training Convergence (ε={epsilon})', fontsize=16, fontweight='bold')
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.savefig('../results/figures/convergence_curves.pdf')
plt.savefig('../results/figures/convergence_curves.png')
plt.show()

## Figure 3: Stability Analysis

In [ ]:
epsilon = 1.0
stability_data = []

for placement in placements:
    placement_df = df[(df['filename'].str.contains(placement)) & 
                      (df['final_epsilon'].between(epsilon-0.1, epsilon+0.1))]
    
    if len(placement_df) > 0:
        stability_data.append({
            'Placement': placement_labels[placement],
            'Gradient Norm Variance': placement_df.iloc[0]['grad_norm_variance'],
            'Loss Oscillation': placement_df.iloc[0]['loss_oscillation']
        })

stability_df = pd.DataFrame(stability_data)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

stability_df.plot(x='Placement', y='Gradient Norm Variance', kind='bar', ax=ax1, legend=False)
ax1.set_ylabel('Gradient Norm Variance', fontsize=14)
ax1.set_title('Training Stability: Gradient Variance', fontsize=14, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)

stability_df.plot(x='Placement', y='Loss Oscillation', kind='bar', ax=ax2, legend=False, color='coral')
ax2.set_ylabel('Loss Oscillation (std)', fontsize=14)
ax2.set_title('Training Stability: Loss Oscillation', fontsize=14, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../results/figures/stability_analysis.pdf')
plt.savefig('../results/figures/stability_analysis.png')
plt.show()

## Figure 4: Systems Efficiency

In [ ]:
epsilon = 1.0
efficiency_data = []

for placement in placements:
    placement_df = df[(df['filename'].str.contains(placement)) & 
                      (df['final_epsilon'].between(epsilon-0.1, epsilon+0.1))]
    
    if len(placement_df) > 0:
        efficiency_data.append({
            'Placement': placement_labels[placement],
            'Avg Epoch Time (s)': placement_df.iloc[0]['avg_epoch_time'],
            'Throughput (samples/s)': placement_df.iloc[0]['avg_throughput']
        })

efficiency_df = pd.DataFrame(efficiency_data)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

efficiency_df.plot(x='Placement', y='Avg Epoch Time (s)', kind='bar', ax=ax1, legend=False, color='steelblue')
ax1.set_ylabel('Time (seconds)', fontsize=14)
ax1.set_title('Average Epoch Time', fontsize=14, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)

efficiency_df.plot(x='Placement', y='Throughput (samples/s)', kind='bar', ax=ax2, legend=False, color='green')
ax2.set_ylabel('Samples/Second', fontsize=14)
ax2.set_title('Training Throughput', fontsize=14, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../results/figures/efficiency_comparison.pdf')
plt.savefig('../results/figures/efficiency_comparison.png')
plt.show()

## Figure 5: MIA Results

In [ ]:
mia_results = []

for file in results_dir.glob('*_mia.json'):
    with open(file, 'r') as f:
        data = json.load(f)
        placement = file.stem.split('_')[2]
        mia_results.append({
            'Placement': placement_labels.get(placement, placement),
            'AUC': data['threshold_attack']['auc'],
            'Advantage': data['threshold_attack']['advantage']
        })

if mia_results:
    mia_df = pd.DataFrame(mia_results)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    mia_df.plot(x='Placement', y='AUC', kind='bar', ax=ax1, legend=False, color='crimson')
    ax1.set_ylabel('Attack AUC', fontsize=14)
    ax1.set_title('Membership Inference Attack: AUC', fontsize=14, fontweight='bold')
    ax1.axhline(y=0.5, color='black', linestyle='--', label='Random Guess')
    ax1.tick_params(axis='x', rotation=45)
    ax1.legend()
    
    mia_df.plot(x='Placement', y='Advantage', kind='bar', ax=ax2, legend=False, color='darkred')
    ax2.set_ylabel('Attack Advantage', fontsize=14)
    ax2.set_title('Membership Inference Attack: Advantage', fontsize=14, fontweight='bold')
    ax2.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig('../results/figures/mia_results.pdf')
    plt.savefig('../results/figures/mia_results.png')
    plt.show()

## Summary Table

In [ ]:
epsilon = 1.0
summary_data = []

for placement in placements:
    placement_df = df[(df['filename'].str.contains(placement)) & 
                      (df['final_epsilon'].between(epsilon-0.1, epsilon+0.1))]
    
    if len(placement_df) > 0:
        row = placement_df.iloc[0]
        summary_data.append({
            'Placement': placement_labels[placement],
            'Accuracy': f"{row['final_accuracy']:.4f}",
            'F1 Score': f"{row['final_f1']:.4f}",
            'Epsilon': f"{row['final_epsilon']:.2f}",
            'Epoch Time (s)': f"{row['avg_epoch_time']:.2f}",
            'Grad Variance': f"{row['grad_norm_variance']:.4f}",
            'Loss Oscillation': f"{row['loss_oscillation']:.4f}"
        })

summary_df = pd.DataFrame(summary_data)
print("\n=== Summary Table (ε=1.0) ===")
print(summary_df.to_string(index=False))

print("\n=== LaTeX Table ===")
print(summary_df.to_latex(index=False))

## Key Findings

1. **Adapter-Only DP** achieves the best privacy-utility tradeoff
2. Applying DP to fewer parameters preserves more utility
3. All placements provide similar empirical privacy protection
4. Training stability varies significantly across placements